# Agents in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q
!pip install llama-index-llms-groq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-openai-like 0.5.3 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.7.9 which is incompatible.
  Using cached llama_index_llms_openai-0.6.26-py3-none-any.whl.metadata (3.0 kB)
Using cached llama_index_llms_openai-0.6.26-py3-none-any.whl (28 kB)
  Attempting uninstall: llama-index-llms-openai
    Found existing installation: llama-index-llms-openai 0.7.9
    Uninstalling llama-index-llms-openai-0.7.9:
      Successfully uninstalled llama-index-llms-openai-0.7.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index 0.14.23 requires llama-index-llms-openai<0.8,>=0.7.0, but you have llama-index-llms-openai 0.6.26 which is incompatible.


## Initialising agents

Let's start by initialising an agent. We will use the basic `AgentWorkflow` class to create an agent.

groq as an inference

In [ ]:
# from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import os
from llama_index.llms.groq import Groq
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream, ReActAgent

os.environ["GROQ_API_KEY"] = "HERE_YOUR_GROQ_API_KEY"



def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b

llm = Groq(model="openai/gpt-oss-120b")

react_agent = ReActAgent(
    name="math",
    description="Math agent",
    llm=llm,
    tools=[subtract, multiply, divide, add],
    system_prompt="You are a math agent...",
    streaming=False,  # Disable LLM streaming
)

agent = AgentWorkflow(agents=[react_agent], root_agent="math")

Then, we can run the agent and get the response and reasoning behind the tool calls.

In [11]:
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print(f"\nCalled tool: {ev.tool_name} {ev.tool_kwargs} => {ev.tool_output}")

resp = await handler
print(resp)


Called tool: add {'a': 2, 'b': 2} => 4
(2 + 2) * 2 = 8


In a similar fashion, we can pass state and context to the agent.


In [12]:
from llama_index.core.workflow import Context

ctx = Context(agent)

response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Your name is Bob.')]), structured_response=None, current_agent_name='math', raw={'id': 'chatcmpl-edd5b08f-2fdf-4aae-b089-e952f5391676', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': "Thought: I can answer without using any more tools. I'll use the user's language to answer\nAnswer: Your name is Bob.", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None, 'reasoning': 'The user asks "What was my name again?" We know from previous message that name is Bob. No tool needed.'}}], 'created': 1782759341, 'model': 'openai/gpt-oss-120b', 'object': 'chat.completion', 'moderation': None, 'service_tier': 'on_demand', 'system_fingerprint': 'fp_c15aa9c1b7', 'usage': {'completion_tokens': 60, 'prompt_tokens': 894, 'total_tokens': 954, 'completion_tokens_de

## Creating RAG Agents with QueryEngineTools

Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. We will pass it to the `AgentWorkflow` class to create a RAG agent.

In [13]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create a query engine
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
llm = Groq(model="openai/gpt-oss-120b")
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. ",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

And, we can once more get the response and reasoning behind the tool calls.

In [14]:
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions."
)
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp


Called tool:  personas {'input': 'science fiction'} => Empty Response

Called tool:  personas {'input': 'science fiction'} => Empty Response

Called tool:  personas {'input': 'science fiction persona'} => Empty Response

Called tool:  personas {'input': 'science'} => Empty Response

Called tool:  personas {'input': 'sci-fi'} => Empty Response

Called tool:  personas {'input': 'fiction'} => Empty Response

Called tool:  personas {'input': 'science fiction characters'} => Empty Response

Called tool:  personas {'input': 'science-fiction'} => Empty Response

Called tool:  personas {'input': 'space'} => Empty Response

Called tool:  personas {'input': 'genre: science fiction'} => Empty Response

Called tool:  personas {'input': 'space explorer'} => Empty Response

Called tool:  personas {'input': 'list'} => Empty Response

Called tool:  personas {'input': 'genre'} => Empty Response
I’m sorry, but there don’t appear to be any persona entries in the database that match a “science‑fiction” q

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='I’m sorry, but there don’t appear to be any persona entries in the database that match a “science‑fiction” query. If you’d like, I can create a few original science‑fiction‑themed persona descriptions for you, or help with something else—just let me know!')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-57a7272e-9176-4c40-8c39-922d3c2d7139', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1782759405, 'model': 'openai/gpt-oss-120b', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': None, 'system_fingerprint': 'fp_acbbc9c4c3', 'usage': {'completion_tokens': 233, 'prompt_tokens': 525, 'total_tokens': 758, 'completion_tokens_details': {'accepted_prediction_tokens': No

## Creating multi-agent systems

We can also create multi-agent systems by passing multiple agents to the `AgentWorkflow` class.

In [15]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
)


# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm,
)

# Create and run the workflow
agent = AgentWorkflow(agents=[calculator_agent, query_agent], root_agent="calculator")

# Run the system
handler = agent.run(user_msg="Can you add 5 and 3?")

In [16]:
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 5, "b": 3}
Called tool:  add {'a': 5, 'b': 3} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The sum of 5 and 3 is 8.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='The sum of 5 and 3 is 8.')]), structured_response=None, current_agent_name='calculator', raw={'id': 'chatcmpl-1d3ee4f8-8b1e-4169-ba56-69304ab9803f', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1782759427, 'model': 'openai/gpt-oss-120b', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': None, 'system_fingerprint': 'fp_4140daa9c2', 'usage': {'completion_tokens': 54, 'prompt_tokens': 883, 'total_tokens': 937, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 12, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.005052084, 'prompt_time': 0.040319885, 'completion_time': 0.115159841, 'total_time': 0.155479